In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit
import boto3

In [16]:
MINIO_ACCESS_KEY= "minioadmin"
MINIO_SECRET_KEY= "minioadmin"
MINIO_ENDPOINT = "http://minio:9000"
BUCKET_NAME= "datalake"
LOCAL_DATA_PATH= "/raw_mount"

In [8]:
# S3_PACKAGES = ",".join([
#     "org.apache.hadoop:hadoop-aws:3.3.6",
#     "com.amazonaws:aws-java-sdk-bundle:1.12.597"
# ])
    # .config("spark.jars.packages", S3_PACKAGES) \
spark= SparkSession.builder.appName("dataToParquet") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Clinvar(Variant Summary)

In [10]:
df_read = spark.read.csv(
    "s3a://datalake/raw/clinvar/variant_summary.txt.gz",
    sep="\t",
    inferSchema=True,
    header=True
)
df_read = df_read.withColumnRenamed("#AlleleID", "AlleleID")
df_read.write.mode("overwrite").parquet("s3a://datalake/raw_parquet/variant_summary_parquet")

NCBI(Gene Info)

In [11]:
df_read = spark.read.csv(
    "s3a://datalake/raw/ncbi/gene_info.gz",
    sep="\t",
    inferSchema=True,
    header=True
)

df_gene = df_read.withColumnRenamed("#tax_id", "tax_id")
df_human = df_gene.filter(col("tax_id") == 9606)

# print(f"Row count for humans: {df_human.count()}")
df_human.write.mode("overwrite").parquet("s3a://datalake/raw_parquet/gene_info_parquet")

DrugBank(Drug-Gene Targets)

In [12]:
df_read = spark.read.csv(
    "s3a://datalake/raw/dgidb/interactions.tsv",
    sep="\t",
    inferSchema=True,
    header=True
)
df_read.write.mode("overwrite").parquet("s3a://datalake/raw_parquet/interactions_parquet")

Opentargets(Gene-Disease)

In [13]:
df_read= spark.read.parquet("s3a://datalake/raw/opentargets/association_by_datasource_direct")
df_read.write.mode("overwrite").parquet("s3a://datalake/raw_parquet/association_parquet")

In [14]:
spark.stop()